
# Evaluation (TEST)



In [ ]:
!pip install seaborn

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


ModuleNotFoundError: No module named 'seaborn'

## 1) Configurar rutas de CSV

In [ ]:

paths = {
    "PPO": "ppo_test.csv",
    "Curriculum": "curriculum_test.csv",
    "Plastic": "plastic_test.csv"
}


## 2) Cargar y combinar datos

In [ ]:

dfs = []

for name, path in paths.items():
    df_i = pd.read_csv(path)
    df_i["method"] = name
    dfs.append(df_i)

df = pd.concat(dfs, ignore_index=True)
df.head()


## 3) Métricas básicas

In [ ]:

metrics = df.groupby("method").agg({
    "success": "mean",
    "episode_reward": ["mean", "std"],
    "steps": "mean",
    "final_distance": "mean"
})

metrics.columns = ["success_rate", "reward_mean", "reward_std", "steps_mean", "distance_mean"]
metrics = metrics.reset_index()
metrics


## 4) Transfer Ratio (TR)

In [ ]:

if "phase" in df.columns:
    results_tr = []
    for method in df["method"].unique():
        d = df[df["method"] == method]
        baseline = d[d["phase"] == "baseline"]["episode_reward"].mean()
        transfer = d[d["phase"] == "transfer"]["episode_reward"].mean()
        tr = transfer / baseline if baseline != 0 else np.nan
        results_tr.append({"method": method, "TR": tr})
    df_tr = pd.DataFrame(results_tr)
    display(df_tr)
else:
    print("⚠️ No 'phase' column → TR no calculado")


## 5) Gráficas

In [ ]:

# Success Rate
plt.figure(figsize=(6,4))
sns.barplot(data=df, x="method", y="success", estimator=np.mean)
plt.title("Success Rate")
plt.show()


In [ ]:

# Final Reward
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="method", y="episode_reward")
plt.title("Final Reward Distribution")
plt.show()


In [ ]:

# Steps
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="method", y="steps")
plt.title("Steps per Episode")
plt.show()


In [ ]:

# Final Distance
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="method", y="final_distance")
plt.title("Final Distance to Goal")
plt.show()


In [ ]:

# Reward vs Episode
plt.figure(figsize=(8,5))
for method in df["method"].unique():
    d = df[df["method"] == method].sort_values("episode")
    smooth = d["episode_reward"].rolling(20).mean()
    plt.plot(d["episode"], smooth, label=method)

plt.legend()
plt.title("Reward over Episodes (Test)")
plt.show()


In [ ]:

# Stability
stability = df.groupby("method")["episode_reward"].std().reset_index()

plt.figure(figsize=(6,4))
sns.barplot(data=stability, x="method", y="episode_reward")
plt.title("Stability (Reward Std)")
plt.show()


## 6) Tabla resumen

In [ ]:

summary = df.groupby("method").agg({
    "success": "mean",
    "episode_reward": "mean",
    "steps": "mean"
}).reset_index()

summary.columns = ["Method", "Success Rate", "Final Reward", "Avg Steps"]
summary
